# Notebook 01: Train a Generative Model for Inverse Design

**Can we learn to skip the optimizer?**

In Notebook 00 you saw that EngiBench bundles an optimizer with every problem.
Running that optimizer produces an optimal design — but it takes time. For
Beams2D it runs in seconds, but for complex 3D problems it can take minutes or
hours *per design*.

Generative AI offers a different approach: **train a neural network once on a
dataset of optimal designs, then generate new designs instantly.** The trade-off
is quality for speed — and the central question of this workshop is *how do we
measure that trade-off rigorously?*

### What you will do

| Step | What happens | Key concept |
|------|-------------|-------------|
| **Prepare data** | Extract conditions and designs from EngiBench | The standardised data API |
| **Train a model** | Fit a neural network to map conditions → designs | Supervised learning on design data |
| **Generate designs** | Produce new designs for unseen conditions | Instant inference vs. slow optimization |
| **Inspect results** | Compare generated vs. ground-truth designs visually | Setting up evaluation (Notebook 02) |

> **Heads up:** We deliberately train a simple model with limited data and few
> epochs. The results will be imperfect — **that is the point.** Understanding
> *why* they are imperfect motivates the rigorous benchmarking we explore in
> Notebook 02 and the discussion session.

---

### Exercise legend
| Marker | Meaning |
|---|---|
| `FILL-IN CELL` | Your turn — edit the code between `START FILL` / `END FILL` |
| `CHECKPOINT` | Automated check — if it fails, fix before moving on |

> **Colab users:** click **File > Save a copy in Drive** before editing so your changes persist.

## 0. Install dependencies

In [ ]:
# Colab / local dependency bootstrap
import subprocess, sys

IN_COLAB = "google.colab" in sys.modules
FORCE_INSTALL = False  # Set True to force install outside Colab

if IN_COLAB or FORCE_INSTALL:
    def _pip(pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])
    _pip(["engibench[beams2d]", "sqlitedict", "matplotlib", "tqdm", "tyro", "wandb"])
    _pip(["git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt"])
    try:
        import torch
    except Exception:
        _pip(["torch", "torchvision"])
    print("Install complete.")
else:
    print("Using current environment. Set FORCE_INSTALL=True to install here.")

---

## The inverse design problem

Traditional topology optimization works like this:

```
Conditions (volfrac, loads, …)  ──►  [ Optimizer (iterative) ]  ──►  Optimal design
                                        ⏱ seconds to hours
```

A **learned generator** replaces the optimizer with a neural network:

```
Conditions  ─┐
              ├──►  [ Neural network ]  ──►  Approximate design
Random noise ─┘        ⏱ milliseconds
```

The noise input lets the model produce **diverse** designs for the same
conditions — useful for exploring the design space. But the designs are only
*approximate*: the network has to generalise from training examples rather than
solving the physics directly.

**Key question:** How close can a learned generator get to the optimizer? That
is what benchmarking measures.

## 1. Imports

In [ ]:
import importlib
import json
import random
import sys, os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch as th

# Workshop helpers (visualization + training utilities)
_utils = os.path.abspath("../utils") if os.path.isdir("../utils") else "workshops/dcc26/utils"
sys.path.insert(0, _utils)
import notebook_helpers  # noqa: E402
importlib.reload(notebook_helpers)  # always pick up latest edits
from notebook_helpers import *  # noqa: F401,F403

from engibench.utils.all_problems import BUILTIN_PROBLEMS
from engiopt.cgan_cnn_2d.cgan_cnn_2d import Generator as EngiOptCNNGenerator

## 2. Configuration

All tuneable knobs in one place. **Experiment with these** — especially
`EPOCHS` and `N_TRAIN` — to see how they affect the generated designs.

In [ ]:
# ---------- Reproducibility ----------
SEED = 7

# ---------- Problem ----------
PROBLEM_ID = "beams2d"  # Change to try a different EngiBench problem

# ---------- Training ----------
EPOCHS     = 15      # Short for workshop; try 50+ for better results
BATCH_SIZE = 64
LR         = 2e-4    # Adam learning rate
LATENT_DIM = 32      # Size of random noise vector fed to generator
# ---------- Generation ----------
N_SAMPLES  = 24      # Designs to generate for Notebook 02

# ---------- Device ----------
if th.cuda.is_available():
    DEVICE = th.device("cuda")
elif th.backends.mps.is_available():
    DEVICE = th.device("mps")
else:
    DEVICE = th.device("cpu")
print("Device:", DEVICE)

# ---------- Artifact paths ----------
ARTIFACT_DIR = Path("/content/dcc26_artifacts") if "google.colab" in sys.modules else Path("workshops/dcc26/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH        = ARTIFACT_DIR / "engiopt_cgan2d_generator_supervised.pt"
HISTORY_PATH     = ARTIFACT_DIR / "training_history.csv"
TRAIN_CURVE_PATH = ARTIFACT_DIR / "training_curve.png"

# ---------- Seed everything ----------
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
if th.cuda.is_available():
    th.cuda.manual_seed_all(SEED)

print("Problem:     ", PROBLEM_ID)
print("Artifact dir:", ARTIFACT_DIR)

---

## 3. Load the EngiBench problem

Same API you used in Notebook 00 — every problem exposes `.dataset`,
`.conditions_keys`, and `.design_space`.

In [ ]:
problem = BUILTIN_PROBLEMS[PROBLEM_ID](seed=SEED)
train_ds = problem.dataset["train"]
test_ds  = problem.dataset["test"]

condition_keys = problem.conditions_keys
design_shape   = problem.design_space.shape
n_conds        = len(condition_keys)

print(f"Problem        : {type(problem).__name__}")
print(f"Design shape   : {design_shape}")
print(f"Condition keys : {condition_keys}")
print(f"Train examples : {len(train_ds)}")
print(f"Test examples  : {len(test_ds)}")

In [ ]:
# Quick look at a few training designs
show_design_gallery(problem.dataset, problem, n=4, seed=SEED)

---

## 4. FILL-IN 01-A: Prepare training data

The EngiBench dataset stores conditions and designs as separate columns.
To train a neural network we need to extract them into numeric arrays:

1. **Conditions**: a `(N, n_conds)` array of floats — one row per sample, one column per condition key
2. **Designs**: a `(N, H, W)` array of pixel values

We use the **full training set** so the model sees as many examples as possible.
We also rescale designs from `[0, 1]` to `[-1, 1]` because the generator uses a
`tanh` output layer (which naturally outputs that range).

In [ ]:
# FILL-IN CELL 01-A
# Goal: extract conditions and designs from the full EngiBench training set.

rng = np.random.default_rng(SEED)

# START FILL ---------------------------------------------------------------

# 1. Stack all condition columns into one (N, n_conds) array
#    Hint: use np.stack with a list comprehension over condition_keys
#    Example: np.stack([np.array(train_ds[k]).astype(np.float32)
#                       for k in condition_keys], axis=1)
conds_np = None

# 2. Extract the optimal designs
#    Hint: np.array(train_ds["optimal_design"]).astype(np.float32)
designs_np = None

# 3. Rescale designs from [0, 1] to [-1, 1]
#    Hint: targets = designs * 2.0 - 1.0
targets_np = None

# END FILL -----------------------------------------------------------------

# CHECKPOINT
n_train = len(train_ds)
assert conds_np is not None and designs_np is not None and targets_np is not None, (
    "Fill in conds_np, designs_np, and targets_np above."
)
assert conds_np.shape == (n_train, n_conds), (
    f"Expected conditions shape ({n_train}, {n_conds}), got {conds_np.shape}"
)
assert targets_np.shape == (n_train, *design_shape), (
    f"Expected targets shape ({n_train}, {', '.join(map(str, design_shape))}), got {targets_np.shape}"
)
assert targets_np.min() >= -1.0 and targets_np.max() <= 1.0, (
    f"Targets should be in [-1, 1], got [{targets_np.min():.2f}, {targets_np.max():.2f}]"
)
print(f"CHECKPOINT passed: {n_train} samples, conditions {conds_np.shape}, targets {targets_np.shape}")

---

## 5. The Generator model

We use a **convolutional conditional generator** (cDCGAN) from EngiOpt. Unlike a
simple fully-connected network that treats the design as a flat vector of pixels,
this model uses **transposed convolutions** that upsample a small feature map
into a full-resolution design image — preserving spatial structure at every step.

```
noise (32, 1, 1) ──► ConvT ──┐
                              ├─► concat (256, 7, 7)
conditions (4, 1, 1) ► ConvT ┘         │
                                        ▼
                              ConvT  7×7  → 13×13
                              ConvT 13×13 → 25×25
                              ConvT 25×25 → 50×50
                              ConvT 50×50 → 100×100 → resize → design
```

This **convolutional inductive bias** is why CNN generators produce much sharper
designs than MLP generators: each layer reasons about local spatial
neighbourhoods rather than treating every pixel independently.

In [ ]:
# Wrap the CNN generator so it accepts flat (B, dim) inputs
from notebook_helpers import WorkshopGenerator

cnn_gen = EngiOptCNNGenerator(
    latent_dim=LATENT_DIM,
    n_conds=n_conds,
    design_shape=design_shape,
)
model = WorkshopGenerator(cnn_gen).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Generator created: {n_params:,} parameters")
print(f"Input:  noise ({LATENT_DIM}) + conditions ({n_conds}) = {LATENT_DIM + n_conds}")
print(f"Output: {' x '.join(map(str, design_shape))} design image")

---

## 6. FILL-IN 01-B: Train the model

Training is **supervised**: for each sample, the model sees random noise +
conditions and tries to reproduce the optimal design. The loss measures
pixel-by-pixel error (MSE).

We provide a `train_supervised_generator()` helper that handles the training
loop. Your job: **call it with the right arguments and experiment with
settings.**

> **Try it:** After training with the default 8 epochs, change `EPOCHS` to 20
> or 50 in the config cell above, re-run from there, and see how the loss and
> designs change.

In [ ]:
# FILL-IN CELL 01-B
# Goal: train the generator. Experiment with EPOCHS and N_TRAIN.

# Pick a few test conditions for snapshot visualization during training
snap_idx = rng.choice(len(test_ds), size=4, replace=False)
snap_conds = np.stack(
    [np.array(test_ds[k])[snap_idx].astype(np.float32) for k in condition_keys],
    axis=1,
)
snap_baselines = np.array(test_ds["optimal_design"])[snap_idx].astype(np.float32)

# START FILL ---------------------------------------------------------------

# Call train_supervised_generator() with appropriate arguments.
# It returns a dict with keys "losses" and "snapshots".
#
# Signature:
#   train_supervised_generator(
#       model, conditions_array, targets_array,
#       latent_dim=..., epochs=..., batch_size=..., lr=..., device=...,
#       snapshot_conditions=..., snapshot_at_epochs=[...],
#   )
#
# Use the variables: model, conds_np, targets_np, LATENT_DIM, EPOCHS,
#   BATCH_SIZE, LR, DEVICE, snap_conds

train_result = None  # Replace with the function call

# END FILL -----------------------------------------------------------------

if train_result is None:
    raise RuntimeError("Call train_supervised_generator() above and assign to train_result.")

train_losses = train_result["losses"]
snapshots = train_result["snapshots"]

# Save checkpoint
th.save(model.state_dict(), CKPT_PATH)

# CHECKPOINT
assert len(train_losses) == EPOCHS, f"Expected {EPOCHS} loss values, got {len(train_losses)}"
assert train_losses[-1] < train_losses[0], (
    "Loss did not decrease — check your training arguments."
)
print(f"\nCHECKPOINT passed: trained for {EPOCHS} epochs, final loss {train_losses[-1]:.6f}")

### Training loss curve

The loss should decrease over epochs. A flat or increasing loss means something
went wrong. Note that even a decreasing loss does not guarantee good designs —
MSE rewards blurry averages.

In [ ]:
# Save training history
import pandas as pd
pd.DataFrame({"epoch": range(1, len(train_losses) + 1), "loss": train_losses}).to_csv(
    HISTORY_PATH, index=False,
)

show_training_curve(train_losses, save_path=str(TRAIN_CURVE_PATH))

### How the generator learns

Below you can see what the generator produces at different points during
training. Early outputs are random noise; later outputs start to resemble beam
structures. The ground-truth row shows what the model is trying to match.

In [ ]:
show_training_progression(snapshots, baseline_designs=snap_baselines, n_show=4)

---

## 7. FILL-IN 01-C: Generate designs from test conditions

Now for the payoff: use your trained model to produce designs for **conditions
it has never seen** (from the held-out test set).

If the model generalises, it should produce reasonable designs for new
conditions without running the optimizer. The `generate_designs()` helper
handles the inference — you just need to:

1. Pick test conditions from the EngiBench dataset
2. Call the generator
3. Also extract the ground-truth baselines for comparison

In [ ]:
# FILL-IN CELL 01-C
# Goal: generate N_SAMPLES designs conditioned on test-set conditions.

# START FILL ---------------------------------------------------------------

# 1. Sample N_SAMPLES indices from the test set
#    Example: test_idx = rng.choice(len(test_ds), size=N_SAMPLES, replace=False)
test_idx = None

# 2. Extract test conditions as (N_SAMPLES, n_conds) array and baseline designs
#    Example:
#        test_conds_np = np.stack(
#            [np.array(test_ds[k])[test_idx].astype(np.float32) for k in condition_keys],
#            axis=1,
#        )
#        baseline_designs = np.array(test_ds["optimal_design"])[test_idx].astype(np.float32)
test_conds_np = None
baseline_designs = None

# 3. Generate designs using generate_designs()
#    Example: gen_designs = generate_designs(model, test_conds_np, latent_dim=LATENT_DIM, device=DEVICE)
gen_designs = None

# 4. Build condition records (list of dicts) for JSON export
#    Example:
#        conditions_records = [
#            {k: float(test_conds_np[i, j]) for j, k in enumerate(condition_keys)}
#            for i in range(N_SAMPLES)
#        ]
conditions_records = None

# END FILL -----------------------------------------------------------------

# CHECKPOINT
for name, val in [("test_idx", test_idx), ("test_conds_np", test_conds_np),
                  ("baseline_designs", baseline_designs), ("gen_designs", gen_designs),
                  ("conditions_records", conditions_records)]:
    assert val is not None, f"Fill in {name} above."
assert gen_designs.shape == baseline_designs.shape, (
    f"Shape mismatch: generated {gen_designs.shape} vs baseline {baseline_designs.shape}"
)
assert len(conditions_records) == N_SAMPLES
assert 0.0 <= gen_designs.min() and gen_designs.max() <= 1.0, (
    f"Generated designs should be in [0, 1], got [{gen_designs.min():.2f}, {gen_designs.max():.2f}]"
)
print(f"CHECKPOINT passed: generated {N_SAMPLES} designs, shape {gen_designs.shape}")

---

## 8. Visual comparison: Generated vs Ground Truth

Each column shows the same conditions. Top row = your model's output; bottom row
= the optimizer's solution from the dataset.

**What to look for:**
- **Blurriness:** generated designs are often blurry because MSE loss averages
  over possible solutions
- **Structure:** do the generated designs have recognisable beam topology (load
  paths, supports)?
- **Condition sensitivity:** do different conditions produce visibly different
  designs, or does the model output the same thing regardless?

In [ ]:
show_gen_vs_baseline(gen_designs, baseline_designs, conditions_records, condition_keys)

---

## 9. Export artifacts for Notebook 02

Notebook 02 needs three files to run its evaluation pipeline:
- `generated_designs.npy` — your model's output
- `baseline_designs.npy` — ground-truth designs from the dataset
- `conditions.json` — the conditions used for generation

In [ ]:
np.save(ARTIFACT_DIR / "generated_designs.npy", gen_designs)
np.save(ARTIFACT_DIR / "baseline_designs.npy", baseline_designs)
with open(ARTIFACT_DIR / "conditions.json", "w") as f:
    json.dump(conditions_records, f, indent=2)

# Verify
required = ["generated_designs.npy", "baseline_designs.npy", "conditions.json"]
missing = [f for f in required if not (ARTIFACT_DIR / f).exists()]
assert not missing, f"Missing: {missing}"
print(f"Exported to {ARTIFACT_DIR}:")
for f in required:
    print(f"  {f}")

---

## Discussion

### What you have seen

You trained a neural network on a few hundred examples for a few epochs and used
it to produce beam designs in milliseconds. The results are imperfect — and that
is exactly the point.

### Questions to think about

1. **Why are the designs blurry?** MSE loss penalises pixel-wise error, which
   rewards the *average* of all plausible designs rather than any single sharp
   one. What alternative losses or model architectures might produce crisper
   output? (Think: adversarial loss, diffusion models, VAEs.)

2. **Does the model respond to conditions?** Compare designs generated for very
   different volume fractions or load distributions. If they all look the same,
   the model may have learned the dataset mean rather than the
   condition → design relationship. What might help? (More training data? More
   epochs? A different architecture?)

3. **From pixels to physics.** A design can *look* reasonable but fail under
   simulation — disconnected material, wrong volume fraction, stress
   concentrations. Notebook 02 will run the physics solver on your generated
   designs and quantify these failures.

4. **The benchmarking motivation.** We do not know how bad these designs are
   until we *measure*. That is the role of a benchmark: providing standardised
   evaluation so we can compare methods, track progress, and avoid fooling
   ourselves with visual inspection alone.

5. **What would you change?** If you had an hour instead of 30 minutes, what
   would you try? More data, more epochs, a different model, a different loss
   function? How would you decide whether it *actually* improved?

---

## Next

Proceed to **Notebook 02** to evaluate your generated designs with physics-based
simulation and compute benchmark metrics. Your exported artifacts are the input.